# 03 - Survival, Uplift, CLV & Segmentation

Four linked analyses:

1. **Survival** - Lifelines Kaplan-Meier and Cox PH on the real Telco data.
2. **Uplift** - CausalML on the real randomized **Hillstrom** e-mail experiment,
   plus a per-customer uplift score for the Telco book of business.
3. **CLV** - discounted cash-flow customer lifetime value.
4. **Segmentation** - KMeans clustering with automatic profile labelling.

Every path is built with `os.path.join`.

In [1]:
import os
import sys

import numpy as np
import pandas as pd


def find_project_root(start: str) -> str:
    current = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(current, "src")) and os.path.isdir(
            os.path.join(current, "config")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("Could not locate the project root (src/ + config/).")
        current = parent


PROJECT_ROOT = find_project_root(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
DIAGRAM_DIR = os.path.join(PROJECT_ROOT, "docs", "diagrams")
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System


In [2]:
from src.data_pipeline.load_dataset import load_telco_data
from src.data_pipeline.feature_engineering import engineer_features

snapshot_path = os.path.join(PROCESSED_DATA_DIR, "telco_churn_processed.csv")
if os.path.exists(snapshot_path):
    df = pd.read_csv(snapshot_path)
    print("Loaded snapshot:", snapshot_path)
else:
    df = engineer_features(load_telco_data())
    print("Built engineered frame from the raw Kaggle dataset.")

df["tenure"] = df["tenure"].clip(lower=1)
print("Shape:", df.shape)

Loaded snapshot: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\data\processed\telco_churn_processed.csv
Shape: (7043, 32)


## 1. Survival analysis (Lifelines)

Kaplan-Meier estimates the survival (retention) curve overall and by contract type;
the Cox proportional-hazards model quantifies how each driver changes the churn
hazard.

In [3]:
from src.survival_analysis.survival_model import SurvivalAnalyzer

survival = SurvivalAnalyzer.run_full_analysis(df)

cox = survival.get_cox_summary()
display(cox[["coef", "exp(coef)", "p"]].round(4))

--- Fitting Kaplan-Meier (overall) ---
  Median survival time (overall): inf months
--- Fitting Kaplan-Meier by Contract ---
  Month-to-month: median survival = 35.0 months
  One year: median survival = inf months
  Two year: median survival = inf months
--- Fitting Cox PH model ---



Cox PH Model Summary:
                             coef  exp(coef)  se(coef)  coef lower 95%  coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%  cmp to          z             p    -log2(p)
covariate                                                                                                                                                                    
SeniorCitizen           -0.062013   0.939871  0.053969       -0.167789        0.043764             0.845532             1.044736     0.0  -1.149052  2.505346e-01    1.996918
MonthlyCharges           0.005832   1.005849  0.002556        0.000821        0.010842             1.000822             1.010901     0.0   2.281287  2.253145e-02    5.471916
Contract_encoded        -1.017950   0.361335  0.089710       -1.193778       -0.842121             0.303074             0.430796     0.0 -11.347088  7.667430e-30   96.719100
InternetService_encoded  0.588580   1.801429  0.074868        0.441842        0.735319             1.555570

[SurvivalAnalyzer] Kaplan-Meier plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\kaplan_meier.png


[SurvivalAnalyzer] Cox coefficients plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\cox_coefficients.png


,coef,exp(coef),p
covariate,,,
SeniorCitizen,-0.0620,0.9399,0.2505
MonthlyCharges,0.0058,1.0058,0.0225
Contract_encoded,-1.0179,0.3613,0.0000
InternetService_encoded,0.5886,1.8014,0.0000
num_services,-0.3162,0.7289,0.0000
auto_pay,-0.6642,0.5147,0.0000
contract_risk,0.8360,2.3071,0.0000


## 2. Uplift modeling (CausalML)

### 2a. Genuine randomized experiment - Hillstrom e-mail campaign

The Hillstrom dataset comes from a **real randomized A/B test**, so the difference
in outcomes between the treated and control groups is a true causal effect. This is
what we use to validate the uplift methodology.

In [4]:
from src.data_pipeline.load_dataset import load_hillstrom_data, prepare_uplift_data
from src.uplift_modeling.uplift_model import UpliftModeler

hillstrom = load_hillstrom_data()
print("Hillstrom shape:", hillstrom.shape)
print(hillstrom["segment"].value_counts())

X_h, treatment_h, y_h = prepare_uplift_data(hillstrom, outcome="visit")
print("\nUplift features:", X_h.shape)
print("Treatment rate: {:.1%}".format(treatment_h.mean()))
print("Visit rate by group:")
print(y_h.groupby(treatment_h).mean().rename(index={0: "control", 1: "treated"}).round(4))

Hillstrom shape: (64000, 12)
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Uplift features: (64000, 9)
Treatment rate: 66.7%
Visit rate by group:
segment
control    0.1062
treated    0.1670
Name: visit, dtype: float64


In [5]:
hillstrom_model = UpliftModeler()
hillstrom_model.fit(X_h, treatment_h, y_h)
uplift_h = hillstrom_model.predict_uplift(X_h)
segments_h = hillstrom_model.get_uplift_segments(X_h, treatment_h, y_h)

print("Uplift engine :", "causalml BaseSClassifier" if hillstrom_model.using_causalml else "Two-Model fallback")
print("Mean uplift   : {:.4f}".format(np.mean(uplift_h)))
print("\nSegment sizes:")
print(segments_h["segment"].value_counts())

hillstrom_model.plot_uplift_distribution(os.path.join(DIAGRAM_DIR, "uplift_distribution.png"))
hillstrom_model.plot_uplift_by_segment(segments_h, os.path.join(DIAGRAM_DIR, "uplift_segments.png"))
hillstrom_model.plot_qini_curve(X_h, treatment_h, y_h, os.path.join(DIAGRAM_DIR, "qini_curve.png"))
print("Saved uplift plots to", DIAGRAM_DIR)

[UpliftModeler] Using causalml BaseSClassifier


Uplift engine : causalml BaseSClassifier
Mean uplift   : 0.0594

Segment sizes:
segment
Lost Cause      46846
Persuadable     16005
Sleeping Dog     1149
Name: count, dtype: int64


[UpliftModeler] Uplift distribution plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\uplift_distribution.png


[UpliftModeler] Uplift segments plot saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\uplift_segments.png


[UpliftModeler] Qini curve saved: C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams\qini_curve.png
Saved uplift plots to C:\Users\WELCOME\OneDrive\Desktop\nexvia\Customer Lifetime Value and Churn Intelligence System\docs\diagrams


In [6]:
# Observed (causal) visit rate per segment, treatment vs control.
observed = (
    segments_h.groupby(["segment", "treatment"])["churn"]
    .mean()
    .unstack()
    .rename(columns={0: "control", 1: "treated"})
)
observed["observed_lift"] = observed["treated"] - observed["control"]
display(observed.round(4))

treatment,control,treated,observed_lift
segment,,,
Lost Cause,0.1018,0.1370,0.0352
Persuadable,0.0977,0.2557,0.1580
Sleeping Dog,0.3875,0.1482,-0.2393


### 2b. Per-customer uplift for the Telco book of business

The Telco dataset has no randomized offer flag, so we attach a *simulated*
retention-offer assignment (`add_synthetic_treatment=True`) purely to drive the
downstream ROI simulator. The Hillstrom experiment above is the genuine causal
validation of the method.

In [7]:
telco_treat = engineer_features(load_telco_data(add_synthetic_treatment=True))

from src.data_pipeline.feature_engineering import prepare_model_data

X_t, y_t = prepare_model_data(telco_treat)
treatment_t = telco_treat["treatment"].loc[X_t.index]

telco_uplift = UpliftModeler()
telco_uplift.fit(X_t, treatment_t, y_t)
telco_uplift.predict_uplift(X_t)
telco_segments = telco_uplift.get_uplift_segments(X_t, treatment_t, y_t)

telco_treat = telco_treat.assign(
    uplift_score=telco_segments["uplift_score"],
    uplift_segment=telco_segments["segment"],
)
print("Uplift segment sizes (Telco):")
print(telco_treat["uplift_segment"].value_counts())

[UpliftModeler] Using causalml BaseSClassifier


Uplift segment sizes (Telco):
uplift_segment
Sleeping Dog    3878
Persuadable     1761
Lost Cause      1404
Name: count, dtype: int64


## 3. Customer lifetime value

Expected remaining tenure comes from the churn probability via the geometric
approximation `E[T] = (1 - p) / p`, and cash flows are discounted at the monthly
discount rate.

In [8]:
from src.churn_model.churn_classifier import ChurnClassifier
from src.clv_segmentation.clv_calculator import CLVCalculator

classifier, metrics, X_test, y_test = ChurnClassifier.train_and_evaluate(telco_treat)
X_all, _ = prepare_model_data(telco_treat)
telco_treat["churn_probability"] = classifier.predict(X_all)

telco_treat = CLVCalculator.compute_all_clv(telco_treat, telco_treat["churn_probability"].to_numpy())

print("Churn AUC: {:.3f}".format(metrics["auc_roc"]))
print("CLV summary (USD):")
print(telco_treat["clv"].describe().round(2))

The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.


Churn AUC: 0.863
CLV summary (USD):
count    7043.00
mean      221.71
std       285.02
min         5.79
25%        28.04
50%        93.85
75%       276.91
max      1291.45
Name: clv, dtype: float64


## 4. Customer segmentation (KMeans)

Customers are clustered on charges, tenure, CLV, service count and churn
probability, then auto-labelled into Champions, At Risk High Value, Loyal Low Value
and Price Sensitive.

In [9]:
from src.clv_segmentation.customer_segments import CustomerSegmenter

segmenter, segmented_df, profiles = CustomerSegmenter.run_segmentation(telco_treat)
display(profiles[[
    "segment", "label", "count", "pct_of_total", "avg_clv",
    "avg_churn_probability", "dominant_contract", "recommended_action",
]].round(3))

,segment,label,count,pct_of_total,avg_clv,avg_churn_probability,dominant_contract,recommended_action
0,0,Champions,2316,32.9,161.90,0.104,Month-to-month,Loyalty programme – upsell & cross-sell
1,1,Champions,1114,15.8,799.11,0.028,Two year,Loyalty programme – upsell & cross-sell
2,2,Price Sensitive,2066,29.3,27.81,0.566,Month-to-month,Value-based discounts – cost-sensitive retention
3,3,Price Sensitive,1547,22.0,154.41,0.228,Month-to-month,Loyalty programme – upsell & cross-sell


## 5. Offer ROI preview

For each customer we combine uplift (incremental retention) with CLV to estimate
the net value of a retention offer. Notebook 04 and the Streamlit app expose the
full discount simulator.

In [10]:
calc = CLVCalculator()
roi = calc.compute_expected_revenue_impact(
    segmented_df, segmented_df["uplift_score"].to_numpy(), offer_cost=10.0
)

rows = []
for cost in [5, 10, 20, 40]:
    r = calc.compute_expected_revenue_impact(
        segmented_df, segmented_df["uplift_score"].to_numpy(), offer_cost=cost
    )
    rows.append({
        "offer_cost": cost,
        "customers_profitable": int((r["net_value"] > 0).sum()),
        "total_net_value": round(r["net_value"].clip(lower=0).sum(), 2),
    })
display(pd.DataFrame(rows))

,offer_cost,customers_profitable,total_net_value
0,5,17,196.91
1,10,16,113.40
2,20,6,32.13
3,40,0,0.00
